In [ ]:
# load index file for all input data (secondary outputs)
# please note.. secondary outputs are stored directly by task (data doesn't have to be archived)
# so if you submitted multiple tasks for the same subject, there could be more than 1 output for 
# that subject. you will need to decide which one to use (sort by finish_date?)
import json
with open("input/index.json") as f:
    index = json.load(f) 
    
print("number of objects", len(index["objects"]))
print("dumping the list of objects")
for object in index["objects"]:
    print(object["meta"]["subject"], object["datatype"]["name"])
    
#you can examine the structure of index.json by opening it on another tab. it's under input/ directory.

In [ ]:
# participant information is also stored in the index.json
# TODO - I think I will move participants information out of index.json
for (subject, info) in index["participants"]["subjects"].items():
    print(info)


In [ ]:
# let's load parc-stats/rois.csv for all subjects.
# while at it, let's group them into gamer v.s non-gamer

import pandas

gamer_rois=[]
nongamer_rois=[]

for object in index["objects"]:
    if object["datatype"]["name"] == "neuro/parc-stats":
        subject=object["meta"]["subject"]
        csv = pandas.read_csv("input/"+object["path"]+"/cortex.csv", index_col="subjectID")
        phenotype = index["participants"]["subjects"][subject]
        if phenotype["group"] == "Gamer":
            gamer_rois.append(csv)
        elif phenotype["group"] == "Non-gamer":
            nongamer_rois.append(csv)
        else:
            print("I don't know what to do with this subject", subject, phenotype["group"])

#convert to a single panda frame for each group
gamer_rois = pandas.concat(gamer_rois)
nongamer_rois = pandas.concat(nongamer_rois)

#show a sample..
gamer_rois

In [ ]:
# now we can perform some statistics
groupby="structureID"

gamer_means = gamer_rois.groupby(groupby).mean()
nongamer_means = nongamer_rois.groupby(groupby).mean()

gamer_std = gamer_rois.groupby(groupby).std()
nongamer_std = nongamer_rois.groupby(groupby).std()


#dump..
print("gamer")
print(gamer_means["average_thickness_mm"])
print("non-gamer")
print(nongamer_means["average_thickness_mm"])

In [ ]:
column="average_thickness_mm"

# visualize the results
df_means = pandas.DataFrame({
    "gamer": gamer_means[column], "non-gamer": nongamer_means[column], 
})
df_std = pandas.DataFrame({
    "gamer": gamer_std[column], "non-gamer": nongamer_std[column], 
})
df_means.plot.bar(figsize=(25,7), yerr=df_std)
